# THIS IS THE FINAL VERSION FOR nnUNET_DOB_SCV
all of the necessary splitting, DOB Straificaion, Cleaning, and ROI Masking has been done before this NB

## Importing all the Dependencies

In [4]:
# !rm -rf /kaggle/working/*

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [5]:
# IMPORT NN UNET

import sys

sys.path.insert(0, "/kaggle/input/notebooks/kalabhalu/nnunet-importer")

# Checking with INTERNT OFF and GPU ON
import nnunetv2

import importlib.metadata as md

print(md.version("nnunetv2"))

2.8.0


## Importing the properly unzipped notebook to process

In [6]:
from pathlib import Path
import random

sys.path.insert(0, "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2")
base = Path("/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2")


files = [f for f in base.rglob("*") if f.is_file()]

for f in random.sample(files, min(10, len(files))):
    print(f.relative_to(base))

nnUNet_raw/Dataset001_CAC/labelsTr/id_248.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_68.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_13.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_198.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_414.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_392.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTr/id_292_0000.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTs/id_378.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/id_353.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTr/id_165_0000.nii.gz


## Environment Variables for nnUnet

In [7]:
import os
import nnunetv2

import importlib.metadata as md

print(md.version("nnunetv2"))

os.environ["nnUNet_raw"] = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"  

!mkdir -p /kaggle/working/nnUNet_raw
!mkdir -p /kaggle/working/nnUNet_preprocessed
!mkdir -p /kaggle/working/nnUNet_results

2.8.0


## Pre Prcoessing for nnUnet

In [8]:
import sys
from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry

sys.argv = [
    "plan_and_preprocess",
    "-d", "1",              # dataset id
    "-npfp", "4",           # fingerprint processes
    "-np", "4",             # preprocessing processes
    "--verify_dataset_integrity"
]

plan_and_preprocess_entry()

Fingerprint extraction...
Dataset001_CAC
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [3.      0.38625 0.38625]. 
Current patch size: (np.int64(28), np.int64(256), np.int64(256)). 
Current median shape: [ 47.         504.85436893 504.85436893]
Attempting to find 3d_lowres config. 
Current spacing: [3.        0.3978375 0.3978375]. 
Current patch size: (np.int64(28), np.int64(256), np.int64(256)). 
Current median shape: [ 47.         490.14987275 

Preprocessing cases: 100%|██████████| 368/368 [00:52<00:00,  6.95it/s]


Configuration: 3d_fullres...
{'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [24, 256, 256], 'median_image_size_in_voxels': [47.0, 520.0, 520.0], 'spacing': [3.0, 0.375, 0.375], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': {'n_stages': 7, 'features_per_stage': [32, 64, 128, 256, 320, 320, 320], 

Preprocessing cases: 100%|██████████| 368/368 [00:51<00:00,  7.17it/s]


Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset001_CAC. Skipping.


## Training nnUNet

In [9]:
import os

os.environ["TORCHDYNAMO_DISABLE"] = "1"    
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton"

import torch
torch._dynamo.config.suppress_errors = True

In [10]:
import sys
from nnunetv2.run.run_training import run_training_entry

sys.argv = [
    "nnUNetv2_train",
    "1",              # dataset id
    "3d_fullres",     # configuration
    "0",              # fold (0–4)
]

run_training_entry()


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-08-27 13:17:43.715526: Using torch.compile...
2026-08-27 13:17:43.746679: do_dummy_2d_data_aug: True
2026-08-27 13:17:43.747281: Using splits from existing split file: /kaggle/working/nnUNet_preprocessed/Dataset001_CAC/splits_final.json
2026-08-27 13:17:43.747429: The split file contains 5 

## Test Set Results (FOLD 0 TRAINED MODEL)

In [15]:
import torch

torch.set_num_interop_threads = lambda *args, **kwargs: None

In [16]:
import sys
from nnunetv2.inference.predict_from_raw_data import predict_entry_point

sys.argv = [
    "nnUNetv2_predict", 
    "-i", "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2/nnUNet_raw/Dataset001_CAC/imagesTs",
    "-o", "/kaggle/working/predictions",
    "-d", "Dataset001_CAC",
    "-c", "3d_fullres",
    "-f", "0",
    "-chk", "checkpoint_best.pth"
]

predict_entry_point()


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 65 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 65 cases that I would like to predict

Predicting id_169:
perform_everything_on_device: True


100%|██████████| 36/36 [00:03<00:00, 10.44it/s]


sending off prediction to background worker for resampling and export
done with id_169

Predicting id_175:
perform_everything_on_device: True


100%|██████████| 27/27 [00:02<00:00, 10.48it/s]


sending off prediction to background worker for resampling and export
done with id_175

Predicting id_182:
perform_everything_on_device: True


...

100%|██████████| 27/27 [00:02<00:00, 10.47it/s]


sending off prediction to background worker for resampling and export
done with id_94

Predicting id_98:
perform_everything_on_device: True


100%|██████████| 36/36 [00:03<00:00, 10.43it/s]


sending off prediction to background worker for resampling and export
done with id_98
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.


In [17]:
import os
import numpy as np
import SimpleITK as sitk

pred_dir = "/kaggle/working/predictions"
gt_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2/nnUNet_raw/Dataset001_CAC/labelsTs"

# Only prediction images
files = sorted([
    f for f in os.listdir(pred_dir)
    if f.endswith(".nii.gz")
])

dice_scores = []

for f in files:

    pred_path = os.path.join(pred_dir, f)
    gt_path = os.path.join(gt_dir, f)

    if not os.path.exists(gt_path):
        print(f"Ground truth not found for {f}, skipping.")
        continue

    pred = sitk.GetArrayFromImage(sitk.ReadImage(pred_path))
    gt = sitk.GetArrayFromImage(sitk.ReadImage(gt_path))

    labels = sorted(set(np.unique(pred)).union(set(np.unique(gt))))
    labels = [l for l in labels if l != 0]  # Ignore background

    case_dices = []

    for label in labels:

        pred_mask = (pred == label)
        gt_mask = (gt == label)

        tp = np.logical_and(pred_mask, gt_mask).sum()
        fp = np.logical_and(pred_mask, ~gt_mask).sum()
        fn = np.logical_and(~pred_mask, gt_mask).sum()

        denom = 2 * tp + fp + fn

        if denom == 0:
            dice = 1.0
        else:
            dice = 2 * tp / denom

        case_dices.append(dice)

    mean_case_dice = np.mean(case_dices) if case_dices else 1.0

    dice_scores.append(mean_case_dice)

    print(f"{f:<35} Dice = {mean_case_dice:.4f}")

print("-" * 60)
print(f"Average Dice = {np.mean(dice_scores):.4f}")

id_169.nii.gz                       Dice = 0.8095
id_175.nii.gz                       Dice = 0.8677
id_182.nii.gz                       Dice = 0.9260
id_193.nii.gz                       Dice = 0.8943
id_203.nii.gz                       Dice = 0.9111
id_206.nii.gz                       Dice = 0.9005
id_214.nii.gz                       Dice = 0.5455
id_215.nii.gz                       Dice = 0.7828
id_226.nii.gz                       Dice = 0.9049
id_237.nii.gz                       Dice = 0.0000
id_240.nii.gz                       Dice = 0.7390
id_243.nii.gz                       Dice = 0.9048
id_244.nii.gz                       Dice = 0.8468
id_246.nii.gz                       Dice = 0.8953
id_249.nii.gz                       Dice = 0.8992
id_26.nii.gz                        Dice = 0.7529
id_261.nii.gz                       Dice = 0.8000
id_266.nii.gz                       Dice = 0.3005
id_273.nii.gz                       Dice = 0.8333
id_276.nii.gz                       Dice = 0.8173


In [18]:
import os
import shutil

pred_dir = "/kaggle/working/predictions"
gt_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2/nnUNet_raw/Dataset001_CAC/labelsTs"
img_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-dob-scv-unzipper-2/nnUNet_raw/Dataset001_CAC/imagesTs"

save_dir = "/kaggle/working/Grouped"
os.makedirs(save_dir, exist_ok=True)

# predictions only
pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith(".nii.gz")])

for pred_file in pred_files:

    # -------- extract id --------
    # prediction: id.nii.gz OR id something
    case_id = pred_file.replace(".nii.gz", "")

    # ground truth format: id.nii.gz
    gt_file = f"{case_id}.nii.gz"

    # image format: id_0000.nii.gz
    img_file = f"{case_id}_0000.nii.gz"

    case_folder = os.path.join(save_dir, case_id)
    os.makedirs(case_folder, exist_ok=True)

    # -------- copy prediction --------
    shutil.copy(
        os.path.join(pred_dir, pred_file),
        os.path.join(case_folder, "prediction.nii.gz")
    )

    # -------- copy GT --------
    gt_path = os.path.join(gt_dir, gt_file)
    if os.path.exists(gt_path):
        shutil.copy(gt_path, os.path.join(case_folder, "label.nii.gz"))
    else:
        print("Missing GT:", gt_path)

    # -------- copy image --------
    img_path = os.path.join(img_dir, img_file)
    if os.path.exists(img_path):
        shutil.copy(img_path, os.path.join(case_folder, "image.nii.gz"))
    else:
        print("Missing image:", img_path)

print("DONE ✔ Grouped dataset created")

DONE ✔ Grouped dataset created
